# Checking for bad (invalid) background events

Events are considered invalid if they have $0$ detected jets.

In [1]:
from pathlib import Path
from diquark.config.config_manager import ConfigManager
from diquark.data.loader import DataLoader

config_files_dir = Path(".").absolute().parent / "diquark" / "config"

# config_file_path = config_files_dir / "New_Features" / "ATLAS_136_S8000_B7500_32j_5f.yaml"
# config_file_path = config_files_dir / "MadGraph_Data" / "MSuu_8000_shatmin_7500.yaml"
# config_file_path = config_files_dir / "New_Features" / "ATLAS_136_S8000_B7500_ptj_10.yaml"
config_file_path = config_files_dir / "New_Features" / "ATLAS_136_S8000_B7500_ptj_100.yaml"

assert config_file_path.exists()

config = ConfigManager(str(config_file_path))

# Load backgrounds config file
backgrounds_config_file = config.get_required("data.backgrounds_file")
backgrounds_config_file_path = (
    config_files_dir / "Backgrounds" / backgrounds_config_file
)
backgrounds_config = ConfigManager(backgrounds_config_file_path)

backgrounds_directory = Path(
    backgrounds_config.get("backgrounds.base_directory", "/")
)
backgrounds_file_names = backgrounds_config.get("backgrounds.file_names", {})
backgrounds_file_name_mapping = backgrounds_config.get(
    "backgrounds.file_name_mapping", {}
)

files = set()

def check_root_file_exists(path: str) -> Path:
    p = Path(path)
    if not p.exists():
        raise Exception(f"Could not find background file at path '{p}'")
    files.add(p)
    return p

background_path_dict = {
    f"BKG:{key}": check_root_file_exists(path)
    for key, path in backgrounds_file_names.items()
}
assert len(files) == len(background_path_dict.keys())

def find_root_file_by_name(filename: str) -> Path:
    results = backgrounds_directory.glob(f"*{filename}*.root")
    for path in results:
        files.add(path)
        return path
    else:
        raise Exception(
            f"Could not find background file with filename '{filename}'"
        )

background_path_dict |= {
    f"BKG:{key}": find_root_file_by_name(filename)
    for key, filename in backgrounds_file_name_mapping.items()
}
assert len(files) == len(background_path_dict.keys())

# Load signal config file
signal_config_file = config.get_required("data.signal_file")
signal_config_file_path = config_files_dir / "Signals" / signal_config_file
signal_config = ConfigManager(str(signal_config_file_path))

signal_file = Path(signal_config.get("signal.file"))
path_dict = background_path_dict | {"SIG:Suu": signal_file}

data_loader = DataLoader(
    path_dict,
    index_start=0,
    index_stop=None,
)

In [2]:
data = data_loader.load_data()

Loading data:   0%|          | 0/30 [00:00<?, ?it/s]

In [3]:
from tqdm.contrib.concurrent import thread_map
from diquark.features.feature_extractor import FeatureExtractor

print("Extracting features...")
feature_extractor = FeatureExtractor(n_jets=32, chi_mass=2000, suu_mass=8000)

features = thread_map(
    feature_extractor.compute_all,
    data.values(),
    max_workers=32,
    desc="Extracting features",
)

print(f"Working with {len(features[0].keys())} feature columns")

assert len(features[0].keys()) == len(feature_extractor.feature_names), (
    f"Number of extracted features ({len(features[0].keys())}) doesn't match number of feature names defined on feature extractor object ({len(feature_extractor.feature_names)})"
)

datasets = dict(zip(data.keys(), features))

Extracting features...


Extracting features:   0%|          | 0/30 [00:00<?, ?it/s]

Working with 73 feature columns


In [10]:
from diquark.data.preprocessor import Preprocessor

preprocessor = Preprocessor(config.get("preprocessing", default={}))
df = preprocessor.create_dataframe(datasets)

In [11]:
df

,jet_multiplicity,p_T_sum,p_T_min,p_T_mean,p_T_stddev,p_T_max,eta_sum,eta_min,eta_mean,eta_stddev,...,chi2_second_component_mean,chi2_second_component_stddev,chi2_second_component_max,chi2_third_component_sum,chi2_third_component_min,chi2_third_component_mean,chi2_third_component_stddev,chi2_third_component_max,Truth,target
0,5,2233.089600,26.626402,446.617920,488.213154,1073.225098,-3.370449,-3.467211,-0.674090,2.419235,...,7407.892187,9716.506231,23714.957031,0.0,0.0,0.0,0.0,0.0,BKG:gg_bbbar,0
1,4,1272.202148,31.212275,318.050537,168.465212,459.124176,3.325223,-3.044809,0.831306,2.277105,...,9024.199219,7504.072792,18692.929688,0.0,0.0,0.0,0.0,0.0,BKG:gg_bbbar,0
2,4,1256.551758,48.892715,314.137939,193.557110,582.092041,1.096540,-2.891194,0.274135,2.218020,...,11734.975586,11685.106247,23618.060547,0.0,0.0,0.0,0.0,0.0,BKG:gg_bbbar,0
3,3,303.840332,27.200499,101.280111,58.736163,170.863159,2.115419,-3.749173,0.705140,3.237992,...,10478.061523,1.646529,10478.061523,0.0,0.0,0.0,0.0,0.0,BKG:gg_bbbar,0
4,4,470.622498,32.515865,117.655624,54.984122,167.081116,-4.400144,-4.154712,-1.100036,2.816614,...,5391.432617,4815.409259,10389.782227,0.0,0.0,0.0,0.0,0.0,BKG:gg_bbbar,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2949983,5,2971.939697,59.060265,594.387939,360.294779,1007.185364,-0.388594,-1.296459,-0.077719,1.519570,...,3708.223047,4395.408176,14396.019531,0.0,0.0,0.0,0.0,0.0,SIG:Suu,1
2949984,4,6079.173340,131.384171,1519.793335,1303.472466,3224.071777,2.003150,-0.419667,0.500787,0.892517,...,5548.641113,5402.225634,11280.202148,0.0,0.0,0.0,0.0,0.0,SIG:Suu,1
2949985,5,3944.099121,38.707993,788.819824,495.220946,1366.784058,-0.918893,-1.438898,-0.183779,0.988688,...,4068.490234,5498.790705,15056.952148,0.0,0.0,0.0,0.0,0.0,SIG:Suu,1
2949986,4,5491.178223,712.745117,1372.794556,583.123150,2017.835571,-1.276189,-1.321704,-0.319047,0.742072,...,6228.317383,4069.462198,12039.392578,0.0,0.0,0.0,0.0,0.0,SIG:Suu,1


In [12]:
invalid = df.jet_multiplicity == 0

In [13]:
print("Number of invalid entries:", invalid.sum())
print("Total number of entries:", len(df))
print("Percentage of invalid entries: {:.2f}%".format(invalid.mean() * 100))

Number of invalid entries: 1
Total number of entries: 2949988
Percentage of invalid entries: 0.00%


In [14]:
invalid_counts = df[invalid].value_counts("Truth")
invalid_counts

Truth
BKG:ffbar_Wgm    1
Name: count, dtype: int64

In [15]:
from sys import stdout

invalid_counts.to_csv(stdout)

Truth,count
BKG:ffbar_Wgm,1
